# View intensity statistics

Plots the per-FOV intensity stats from `fov_scheduler.ipynb`. For each bit
(color of each hyb round):

1. **Z-profile** -- median intensity vs z, one line per FOV.
2. **FOV heatmap** -- median intensity of each FOV on the stage-position grid.

Bead channel (488 nm) and bead-plane frames are excluded.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.colors as mcolors

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/measure_stats/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress        import ProgressTracker
from MERci.analysis.view_intensity_stats import load_stats_with_annotations
from MERci.plots.view_intensity_stats_plots import plot_z_profiles, plot_fov_heatmap
from MERci.acquisition.configs import data_frame_mask

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"          # same as in fov_scheduler

# ── Plotting parameters ────────────────────────────────────────────────
# Which rounds to show (None = all rounds with completed data)
SHOW_ROUNDS  = None

# Which color channels to show in nm (None = all non-bead channels)
# e.g. [560, 650, 750] to show only the MERFISH bits channels
SHOW_COLORS  = None

# Only for older frame tables without a `segment` column, whose bead frames are
# told apart by z and colour.
BEAD_Z       = 0.0
BEAD_COLOR   = 488.0

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Image suffix : {IMAGE_SUFFIX}")

In [ ]:
config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                   config.data_dir, image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")

## 3 — Load stats data

Reads all completed stats CSVs and merges them with frame-table information (color, z position).
Results are stored in `stats_df` — a single DataFrame with one row per (FOV, frame).

In [ ]:
stats_df = load_stats_with_annotations(config, meta, tracker)

if stats_df.empty:
    print("No stats data found. Run nb05 first to generate stats CSVs.")
else:
    n_fovs_loaded = stats_df.groupby(["round_id", "fov_id"]).ngroups
    print(f"Loaded stats for {n_fovs_loaded} FOV×round combinations.")
    if "color" in stats_df.columns:
        colors = sorted(stats_df["color"].dropna().unique().astype(int))
        print(f"Colors found   : {colors}")

## 4 — Z-profiles

Median intensity vs z for each bit, one line per FOV.

In [ ]:
if stats_df.empty:
    print("No data to plot.")
elif "color" not in stats_df.columns:
    print("Column 'color' missing — frame table was not found for this experiment.")
else:
    rounds = sorted(stats_df["round_id"].unique())
    if SHOW_ROUNDS is not None:
        rounds = [r for r in rounds if r in SHOW_ROUNDS]

    for rid in rounds:
        rdata   = stats_df[
            (stats_df["round_id"] == rid)
            & data_frame_mask(stats_df, BEAD_Z)
        ]
        colors  = sorted(rdata["color"].dropna().unique())
        if "segment" not in rdata.columns or rdata["segment"].isna().all():   # older frame table
            colors  = [c for c in colors if round(c) != round(BEAD_COLOR)]
        if SHOW_COLORS is not None:
            colors = [c for c in colors if int(round(c)) in SHOW_COLORS]

        for c in colors:
            plot_z_profiles(stats_df, rid, c, BEAD_Z, BEAD_COLOR)

## 5 — FOV intensity heatmaps

Each FOV's median intensity (over z) on the stage-position grid. Brighter =
more signal.

In [ ]:
if stats_df.empty:
    print("No data to plot.")
elif "color" not in stats_df.columns:
    print("Column 'color' missing — frame table was not found for this experiment.")
else:
    rounds = sorted(stats_df["round_id"].unique())
    if SHOW_ROUNDS is not None:
        rounds = [r for r in rounds if r in SHOW_ROUNDS]

    for rid in rounds:
        rdata  = stats_df[
            (stats_df["round_id"] == rid)
            & data_frame_mask(stats_df, BEAD_Z)
        ]
        colors = sorted(rdata["color"].dropna().unique())
        if "segment" not in rdata.columns or rdata["segment"].isna().all():   # older frame table
            colors = [c for c in colors if round(c) != round(BEAD_COLOR)]
        if SHOW_COLORS is not None:
            colors = [c for c in colors if int(round(c)) in SHOW_COLORS]

        for c in colors:
            plot_fov_heatmap(stats_df, meta, rid, c, BEAD_Z, BEAD_COLOR)